In [0]:
from pyspark.sql.functions import *

In [0]:
class batchWC():
    def __init__(self):
        self.base_dir_loc = "/Workspace/Users/dopetechied@gmail.com/SparkStreaming"
        self.base_vol_loc = "/Volumes/streamingdata/dbo"

    def getRawData(self):
        lines = spark.read.format("text")\
            .option("lineSep", ".")\
            .load(f"{self.base_vol_loc}/data/*.txt")
        
        return lines.select(explode(split(lines.value, " ")).alias("word"))
    
    def getQualityData(self, rawDF):
        return rawDF.select(lower(trim(rawDF.word)).alias("word"))\
                .filter(col("word") != " ")\
                .filter(col("word").rlike('[a-z]'))
    
    def getWordCount(self, qualityDF):
        return qualityDF.groupBy("word").count()
    
    def overwriteWordCount(self, wordCountDF):
        return wordCountDF.write\
                .format("delta") \
                .mode("overwrite") \
                .saveAsTable("streamingdata.dbo.word_count_table")
    
    def wordCount(self):
        print(f"\tExecuting Word Count...", end = "")
        rawDF = self.getRawData()
        qualityDF = self.getQualityData(rawDF)
        resultDF = self.getWordCount(qualityDF)
        self.overwriteWordCount(resultDF)
        print("Done")
        




In [0]:
class streamWC():
    def __init__(self):
        self.base_dir_loc = "/Workspace/Users/dopetechied@gmail.com/SparkStreaming"
        self.base_vol_loc = "/Volumes/streamingdata/dbo"

    def getRawData(self):
        lines = spark.readStream.format("text")\
            .option("lineSep", ".")\
            .load(f"{self.base_vol_loc}/data/*.txt")
        
        return lines.select(explode(split(lines.value, " ")).alias("word"))
    
    def getQualityData(self, rawDF):
        return rawDF.select(lower(trim(rawDF.word)).alias("word"))\
                .filter(col("word") != " ")\
                .filter(col("word").rlike('[a-z]'))
    
    def getWordCount(self, qualityDF):
        return qualityDF.groupBy("word").count()
    
    def overwriteWordCount(self, wordCountDF):
        return wordCountDF.writeStream\
                .format("delta") \
                .option("checkpointLocation", f"{self.base_vol_loc}/checkpoint/word_count") \
                .outputMode("complete") \
                .trigger(availableNow=True) \
                .toTable("streamingdata.dbo.word_count_table")
    # Streaming application always require a checkpointLocation which spark already gives us the option of. We just need to define the location of the checkpointLocation
    # Very important and required
    def wordCount(self):
        print(f"\tStarting Word Count Stream...", end = "")
        rawDF = self.getRawData()
        qualityDF = self.getQualityData(rawDF)
        resultDF = self.getWordCount(qualityDF)
        sQuery = self.overwriteWordCount(resultDF)
        print("Done")
        return sQuery
        


